In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

# Define custom model class to match saved models
class PINN_NeuralNet(tf.keras.Model):
    def __init__(self, output_dim=1, num_hidden_layers=4, num_neurons_per_layer=20, activation='tanh', kernel_initializer='glorot_normal', **kwargs):
        super().__init__(**kwargs)
        self.num_hidden_layers = num_hidden_layers
        self.output_dim = output_dim
        
        # Re-create architecture
        self.hidden = [tf.keras.layers.Dense(num_neurons_per_layer,
                             activation=tf.keras.activations.get(activation),
                             kernel_initializer=kernel_initializer)
                           for _ in range(self.num_hidden_layers)]
        self.out = tf.keras.layers.Dense(output_dim)
    
    def call(self, X):
        Z = X
        for i in range(self.num_hidden_layers):
            Z = self.hidden[i](Z)
        return self.out(Z)
    
    # Adding get_config helps avoid warning messages, though not strictly mandatory if you define the class
    def get_config(self):
        config = super().get_config()
        return config

# Load model and normalization vector
save_dir = r"C:\Users\alanh\PunLab\mfnn\models\pig_negcon"

# Dictionary to tell Keras what "PINN_NeuralNet" is
custom_objects = {"PINN_NeuralNet": PINN_NeuralNet}

print("Loading models...")
model_LF = tf.keras.models.load_model(os.path.join(save_dir, 'model_LF'), custom_objects=custom_objects)
model_HF_nl = tf.keras.models.load_model(os.path.join(save_dir, 'model_HF_nl'), custom_objects=custom_objects)
model_HF_l = tf.keras.models.load_model(os.path.join(save_dir, 'model_HF_l'), custom_objects=custom_objects)

# %%
custom_G0 = 0.0075        # Oscillation amplitude (strain)
custom_omega = .628     # Angular frequency (rad/s)
num_points = 200       # Resolution of the curve

# 1. Generate kinematic data for one full cycle (0 to 2*pi/omega)
t_custom = np.linspace(0, 2 * np.pi / custom_omega, num_points) # Fixed to 2*pi for a clean single loop
st_custom = custom_G0 * np.sin(custom_omega * t_custom)
sr_custom = custom_G0 * custom_omega * np.cos(custom_omega * t_custom)
w_custom = np.full_like(t_custom, custom_omega)
g0_vec_custom = np.full_like(t_custom, custom_G0)

# 2. Prepare and Normalize Input
# CRITICAL FIX: Explicitly cast to float32 to match the model's training dtype
X_custom = np.column_stack([st_custom, sr_custom, w_custom, g0_vec_custom]).astype('float32')

# Use the normalize_data function defined earlier
y_placeholder = np.zeros((num_points, 1), dtype='float32') 
X_custom_norm, _ = normalize_data(X_custom, y_placeholder, norm)

# 3. Predict using MFNN
y_LF_custom = model_LF(X_custom_norm)

# CRITICAL FIX: Use tf.concat instead of np.column_stack
# This keeps everything as Tensors and prevents implicit casting/conversion errors
X_MF_custom = tf.concat([X_custom_norm, y_LF_custom], axis=1)

# Predict HF
y_MF_custom = model_HF_nl(X_MF_custom) + model_HF_l(X_MF_custom)

# 4. Denormalize Stress
y_MF_custom_denorm = y_MF_custom.numpy() * norm[-1]

# 5. Plotting
plt.figure(figsize=(8, 6))
plt.plot(st_custom, y_MF_custom_denorm, color='tab:blue', lw=3, label=f'MFNN Prediction')
plt.title(f'Predicted Stress-Strain Loop\n($\gamma_0$={custom_G0}, $\omega$={custom_omega} rad/s)')
plt.xlabel('Strain $\gamma$')
plt.ylabel('Stress $\sigma$ (Pa)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()


Loading models...





NameError: name 'normalize_data' is not defined